# Lab03K — Porte e liquidez: fechamento dos pareceres

**Único notebook novo a executar.** Faça uma cópia no Drive, use CPU e execute tudo. Este lote reúne 24 testes conjuntos, sensibilidades,6.000 simulações e oráculos. Reutiliza exports I/J já existentes; não repete os labs anteriores nem baixa cotações. A calibração pode demorar alguns minutos e imprime progresso a cada 250 réplicas.

Amostras80/117, unidade ativo/classe, pisos e ausências permanecem. O teste conjunto não identifica dois efeitos separados ou causalidade. Os ICs são nominais, com subcobertura material nas simulações. A família J de 16 testes é preservada.


In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import urlretrieve
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
from IPython.display import display,Image
LOCAL=bool(os.environ.get('ALEM_BETA_LOCAL'));BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03k')).resolve();BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ');RUN=BASE/('run_'+STAMP);RUN.mkdir();ROOT=RUN/'runtime';ROOT.mkdir()
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def script(name,*args):subprocess.run([sys.executable,str(ROOT/'scripts'/name),*map(str,args)],check=True)
print('Lab03K | CPU |','integração local, não execução do autor' if LOCAL else 'Colab','|',RUN.name)


## Fontes e exports autenticados

Código público é fixado por commit e SHA256. Os ZIPs anteriores são lidos como dados, sem executar seus scripts. Primeiro procuram-se cópias nas pastas anteriores do Drive; apenas na ausência de hash válido é transferido o mesmo arquivo existente no Drive. Nenhum cache completo de cotações é baixado.


In [ ]:
SOURCE_COMMIT='276ebc08414f01b2363a5c4e554fb79b2a6a2dca'
SOURCE_PINS={'data/lab03i/review_reference/market_returns_2015_2025.csv.gz': 'cdee511530b44db995fb8513c4931717c69498e458dd90495caa40261a4a8427', 'data/lab03i/review_reference/universo_2015.csv': '319ba3c0faae6d9488b9a3fd58fbbbefd0cdd73c8492c826f888ad5536a9f595', 'data/lab03k/protocol_v1.json': 'eb29d28dd448ce7489d73d3cd17843fc9427d43fe2751d0d828c22243c7d764d', 'results/tables/lab03k_20261025/calibration_replications.csv.gz': '3503e9b93f09a8a37437d079848bb57367a783aec9c45fadded42196314f3a95', 'results/tables/lab03k_20261025/calibration_summary.csv': '9fd2e31e71bf7445a30bd9bfc1e63b7e7b078e92070902c9f98a51cf8e94ad37', 'results/tables/lab03k_20261025/coefficients.csv': 'f30d21900294467ed003dde2b625b6c4f6186262c69009f3de502c770990bc4b', 'results/tables/lab03k_20261025/covariates.csv': 'd3be1270dab4c6cb8a049dad558b0cf940b402949e278bace76957b3fe35ef29', 'results/tables/lab03k_20261025/design_diagnostics.csv': 'e2994c4b5780f4577ea0036856bb74b5ddaf33c20214890d8c59ebcad0aee2ee', 'results/tables/lab03k_20261025/eligibility.csv': '7907785e6b837a0eb6fadbd7bda180d5590269896059585e557c8c6cf5a67998', 'results/tables/lab03k_20261025/joint_tests.csv': '4d0431f27fe2c5bbaaf089f72e6892e92c5ca8215aa856c6d28b7d7172ae2987', 'results/tables/lab03k_20261025/succession_inventory.csv': '298ad50a69cb1b631bae4611750189cd78899f3f080cc8cc87bf57ebac100f30', 'scripts/build_lab03k_notebook.py': 'cde3d534d89047b5a40f909f48490a09efce8b0b3eb1f9d6e8d05f57fab9747c', 'scripts/build_lab03k_paper.py': '712a31fb1e7a23728d59bcfd7d1faae6d4f87c6f546215be3b210a5628e62897', 'scripts/calibrate_lab03k.py': '424fc785da6d91d45a64d9b0e9a04a8c058e35a463a8e80220f24d03bbb532b1', 'scripts/run_lab03k.py': '9a6ff4e7c535116d8c5aee671b6d310ad154e0e31c1747bd25fcd5a07b393f2e', 'scripts/verify_lab03k.py': 'd10acebd313a8af46d7cff78cb1378ecef4ad9ef21f7220b674112f95ba66c55', 'src/lab03j.py': 'bc4e32f66814e5bddcfec809591415b8ea765ae443093faba555768c42629788', 'src/lab03k.py': '5490434e4fafd9034f679a5daa5e94794d4a6c708ba5118f10757ae9d97bbc9b', 'tests/test_lab03k.py': '55070bfb17795a750df502bdc62118b5df35c565105028ae82a088cf3400cb28', 'data/lab03k/inputs_manifest.json': '67a5c2ab5dc1d8d031851d436571794795c7553ee029d4ee38d3cac1a6376877'}
for name,sha in SOURCE_PINS.items():
    target=ROOT/name;target.parent.mkdir(parents=True,exist_ok=True)
    if LOCAL:shutil.copy2(Path(os.environ['ALEM_BETA_LOCAL_SOURCE'])/name,target)
    else:urlretrieve('https://raw.githubusercontent.com/robsondf/alem-do-beta-gemini/'+SOURCE_COMMIT+'/'+name,target)
    assert digest(target)==sha,'Fonte divergente: '+name
I_ID='1L5w3uJvjTU4XiQir1BGMF3XNCWLS-_Ru';I_SHA='ff67a56f9b9cc8b91f11bcbcad311d6253fcfe95aef3f9e02303bad677b41e0e'
J_ID='17DxLLmCd-UD0NXlFXDQNANbk1fxmqZ2q';J_SHA='96b56d0a9e3d8ace52f1f4a6b0307c769547e2bfe355a6f6318e6656ab5f8dce'
if LOCAL:
    I_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_I']);J_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_J'])
    assert digest(I_ARCHIVE) in [I_SHA,'2f61beb15aac8638cf92dae0eef58de8bc70cded3a6fe833519171f21f554f06']
    assert digest(J_ARCHIVE)==J_SHA
else:
    from google.colab import auth,drive
    auth.authenticate_user();drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False)
    DRIVE=Path('/content/drive/MyDrive/alem_do_beta');CACHE=DRIVE/'lab03k_cache';CACHE.mkdir(parents=True,exist_ok=True)
    def existing(file_id,expected,filename,candidate_folders):
        candidates=[CACHE/filename]
        for folder in candidate_folders:candidates+=list((DRIVE/folder).glob('*.zip'))
        for q in candidates:
            if q.is_file() and digest(q)==expected:print('Reutilizado:',q.name);return q
        target=CACHE/filename;part=target.with_suffix('.part')
        with part.open('wb') as stream:
            dl=MediaIoBaseDownload(stream,service.files().get_media(fileId=file_id));done=False
            while not done:_,done=dl.next_chunk()
        assert digest(part)==expected,'Export divergente';part.replace(target);return target
    I_ARCHIVE=existing(I_ID,I_SHA,'export_I.zip',['lab03i_resultados','lab03j_cache'])
    J_ARCHIVE=existing(J_ID,J_SHA,'export_J.zip',['lab03j_resultados'])
def extract_selected(archive,items):
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None;manifest=json.loads(z.read('manifesto_exports.json'))
        for member,target in items:
            b=z.read(member);assert hashlib.sha256(b).hexdigest()==manifest[member],member
            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(b)
I_OUTPUT=RUN/'input_I';J_BATCH=RUN/'input_J';required=['market_private.csv.gz','gaps_preserved/returns_private.csv.gz']
extract_selected(I_ARCHIVE,[('outputs_I/'+name,I_OUTPUT/name) for name in required])
jrequired=['outputs_J/covariates_used.csv','outputs_J/complete_case_audit.csv','outputs_J/event_audit_83.csv','outputs_J/controlled_regressions.csv','private_inputs/turnover_private.csv.gz']
extract_selected(J_ARCHIVE,[('batch_J/'+name,J_BATCH/name) for name in jrequired])
print('Fontes:',len(SOURCE_PINS),'| I/J autenticados | nenhuma cotação nova')


## Execução de todo o lote

Bloco 21 principal e 5/63 sensibilidades. Holm 2 para duas nulas conjuntas de referência, Holm 6 para três sensibilidades em duas coortes; separados por bloco. ICs e p individuais são nominais. Beta 2015 reestimado em estrato próprio; setor mantém classificação arquivada, não point-in-time. A exclusão de ativos é retrospectiva e não muda as amostras principais. Giro é reparametrização exata.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03k.py','-v'],check=True)
OUTPUT=RUN/'outputs_K'
script('run_lab03k.py','--i-output',I_OUTPUT,'--j-batch',J_BATCH,'--output',OUTPUT)
script('calibrate_lab03k.py','--i-output',I_OUTPUT,'--j-batch',J_BATCH,'--output',OUTPUT)
script('verify_lab03k.py','--i-output',I_OUTPUT,'--j-batch',J_BATCH,'--output',OUTPUT)
script('build_lab03k_paper.py','--source',OUTPUT,'--paper',RUN/'paper')
count=0;thresholds=0
for expected in sorted((ROOT/'results/tables/lab03k_20261025').glob('*.csv*')):
    actual=pd.read_csv(OUTPUT/expected.name,float_precision='round_trip');reference=pd.read_csv(expected,float_precision='round_trip')
    pd.testing.assert_frame_equal(actual,reference,check_exact=False,atol=1e-10,rtol=1e-9);count+=actual.size
    for col in actual.select_dtypes(include='number'):
        if col=='p' or col.startswith('p_'):
            np.testing.assert_array_equal(actual[col].le(.05),reference[col].le(.05));thresholds+=len(actual)
for name,sha in SOURCE_PINS.items():assert digest(ROOT/name)==sha,name
for name in ['independent_verification.json','calibration_validation.json']:assert json.loads((OUTPUT/name).read_text())['passed']
receipt=dict(passed=True,environment='local_notebook_integration' if LOCAL else 'author_Colab',source_commit=SOURCE_COMMIT,source_files=len(SOURCE_PINS),compared_values=count,p_threshold_positions=thresholds,atol=1e-10,rtol=1e-9,I_sha256=digest(I_ARCHIVE),J_sha256=digest(J_ARCHIVE),python=sys.version,numpy=np.__version__,pandas=pd.__version__)
(OUTPUT/'notebook_comparison.json').write_text(json.dumps(receipt,indent=2)+chr(10));print(json.dumps(receipt,indent=2))


In [ ]:
display(pd.read_csv(OUTPUT/'joint_tests.csv').query('block==21'))
display(pd.read_csv(OUTPUT/'coefficients.csv').query('block==21 and parameter in ["liquidity","size"]'))
display(pd.read_csv(OUTPUT/'calibration_summary.csv'))
display(Image(filename=str(RUN/'paper/figures/lab02/lab03k_controls.png')))
print('Rejeição conjunta não identifica dois efeitos separados. Cobertura nominal não é garantia de 95%.')


## Salvar e compartilhar

O ZIP inclui código, resultados, figura, máscaras e bancos privados para auditoria. **Mantenha o ZIP no Drive; não publique matrizes ou bancos no Git. Salve esta cópia executada e compartilhe seu link.** Homologação econômica, causalidade, sobrevivência e quantidade exata de ações permanecem limites; execução sem exceções não os certifica. Nenhum PR é mergeado por este notebook.


In [ ]:
export=RUN/('resultados_lab03k_'+STAMP+'.zip');manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for base,prefix in [(ROOT,'runtime'),(OUTPUT,'outputs_K'),(RUN/'paper','paper')]:
        for path in sorted(base.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                name=prefix+'/'+path.relative_to(base).as_posix();z.write(path,name);manifest[name]=digest(path)
    z.writestr('manifesto_exports.json',json.dumps(manifest,indent=2))
with zipfile.ZipFile(export) as z:
    assert z.testzip() is None;saved=json.loads(z.read('manifesto_exports.json'));assert set(saved)==set(z.namelist())-{'manifesto_exports.json'}
    for name,sha in saved.items():assert hashlib.sha256(z.read(name)).hexdigest()==sha,name
print('Export:',export.name,'| bytes:',export.stat().st_size,'| SHA256:',digest(export))
if not LOCAL:
    folder=Path('/content/drive/MyDrive/alem_do_beta/lab03k_resultados');folder.mkdir(parents=True,exist_ok=True);destination=folder/export.name;shutil.copy2(export,destination);assert digest(destination)==digest(export);print('SALVO NO DRIVE:',destination)
else:print('Integração local concluída; execução do autor no Colab ainda distinta.')
